In [7]:
import ot
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
import math
import scipy
from scipy.spatial import cKDTree
from scipy import integrate
from scipy import interpolate
from scipy.spatial import Delaunay
from scipy.spatial import Voronoi
from itertools import product
import torch
torch.manual_seed(100000)
import torch.nn as nn
import torch.optim as optim


In [8]:
#settings
start = np.array((0., 1., 1.05))
dt = 0.01
subset_size = 20#cell number
slope =0.1#(\epsilon = 1/slope in paper)parameter in weight computation
traj_length = int(1e6)#length of trajectory
noise_level = 0.25
sample_size = int(500)
epsilon = 1e-4
simulation_t =int(2e4)
steps = 5
N_iters = 30000
invert_variables = [0]
costs1W, costs2W, costs1L, costs2L = [],[],[],[]

In [9]:

# Define the Lorenz-63 System
def Lorenz(x):
    x = x.T
    dx = 10 * (x[1] - x[0])
    dy = x[0] * (28 - x[2]) - x[1]
    dz = x[0] * x[1] - (8/3) * x[2]
    f = np.array((dx, dy, dz)).T
    return f
def Lorenz_vec(x):
    dx = 10 * (x[:,1] - x[:,0])
    dy = x[:,0] * (28 - x[:,2]) - x[:,1]
    dz = x[:,0] * x[:,1] - (8/3) * x[:,2]
    return torch.cat((dx.reshape(len(x),1),dy.reshape(len(x),1),dz.reshape(len(x),1)),dim = 1)
 #trajectory simulation (Euler's method)
def traj(length, starts = start, model = Lorenz):
    x = starts
    traj = np.zeros((length,len(starts)))
    traj[0] = np.array(x)
    for timestep in range(length-1):

        x = x + dt * model(x)

        traj[timestep+1] = np.array(x)

    return traj

def invariant_measure(matrix):
    N = len(matrix)
    rhs = (-1) * (epsilon/N) * torch.ones(N)
    rho = torch.linalg.solve(((1 - epsilon) * matrix - torch.eye(N)),rhs)
    return rho

relu = nn.ReLU()
def decay(x):
    return relu(1-slope*x)
def w(xs):
    dists =  torch.cdist(xs, Voronoi_centers, p =2)
    pre_w = decay(dists)
    return  pre_w / torch.sum(pre_w, dim=1,keepdim = True)


def Ulam(points,Tpoints):
    mat = torch.zeros((subset_size, subset_size))
    #before normalization
    with torch.no_grad():
          randpts_idxs = torch.tensor(tree.query(points.detach().numpy())[1], dtype=torch.int)
    weights = w(Tpoints)
    mat.index_add_(0, randpts_idxs, weights)
    mat = (mat.T)/mat.sum(dim = 1)
    return mat

class W2Loss(torch.autograd.Function):
    @staticmethod
    def forward(ctx, im_net):

        a = np.asarray(im_true_np, dtype=np.float64)
        b = im_net.detach().cpu().numpy().astype(np.float64)
        total_a, total = a.sum(), b.sum()
        a = a / total_a
        b = b / total
        _, log = ot.emd(a, b, MEASURE_OT_COST, log=True)

        # Dual potentials differentiate the cost with respect to normalized b.
        # Apply the chain rule through b = im_net / im_net.sum().
        dual = log["v"]
        gradient = (dual - np.dot(dual, b)) / total
        ctx.save_for_backward(torch.as_tensor(
            gradient, dtype=im_net.dtype, device=im_net.device
        ))
        return im_net.new_tensor(log["cost"])

    @staticmethod
    def backward(ctx, grad_output):
        gradient, = ctx.saved_tensors
        return grad_output * gradient

In [10]:

trajectory_clean = traj(traj_length)

trajectory = trajectory_clean + np.random.normal(0,noise_level,((traj_length,3)))#loooooooong trajectory
trajectory = trajectory[int(1e4):]

In [13]:
    rand_idxs = np.random.choice(len(trajectory), size=sample_size, replace=False)
    observed = trajectory[rand_idxs]
    randpts = torch.tensor(trajectory[rand_idxs],dtype = torch.float)
    Trandpts = torch.tensor(trajectory[rand_idxs+steps],dtype = torch.float)
#normalization
    M_scale = torch.max(torch.abs(randpts))
    randpts /= M_scale
    Trandpts /= M_scale
#Voronoi cell center
    Voronoi_centers = MiniBatchKMeans(n_clusters=subset_size).fit(randpts).cluster_centers_
    MEASURE_OT_COST = ot.dist(Voronoi_centers, Voronoi_centers, metric="sqeuclidean")
    tree = cKDTree(Voronoi_centers)
    
    Voronoi_centers = torch.tensor(Voronoi_centers,dtype = torch.float)
    U_true = Ulam(randpts,Trandpts)
    im_true = invariant_measure(U_true)
    im_true_np = im_true.detach().cpu().numpy()

/homes/yinonghyn/.local/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)


In [6]:
torch.max(torch.abs(randpts))

tensor(1.)

In [ ]:
for iteration in range(10):
    net1 = nn.Sequential(
    nn.Linear(3, 100),
    nn.Tanh(),
    nn.Linear(100, 100),
    nn.Tanh(),
    nn.Linear(100, 100),
    nn.Tanh(),
    nn.Linear(100, 3)
)
    optimizer1 = optim.Adam(net1.parameters(), lr=1e-3)
    net1.train()
    loss1 = []
# Store the first loss
    net1_randpts = randpts.clone()  # safe copy, avoid modifying original tensor
    for _ in range(steps):
        V_field = Lorenz_vec(net1_randpts*M_scale)/M_scale
        V_field[:,invert_variables] = net1(net1_randpts)[:,invert_variables]
        net1_randpts = net1_randpts+dt* V_field
    U_net = Ulam(randpts,net1_randpts)
    im_net = invariant_measure(U_net)
    #initial_L1 = torch.linalg.norm(im_net - im_true, dtype=torch.float32)
    initial_L1 = W2Loss.apply(im_net)
    for i in range(N_iters):
        optimizer1.zero_grad()
        net1_randpts = randpts
        for _ in range(steps):
            V_field = Lorenz_vec(net1_randpts*M_scale)/M_scale
            V_field[:,invert_variables] = net1(net1_randpts)[:,invert_variables]
            net1_randpts = net1_randpts+dt* V_field
        U_net = Ulam(randpts,net1_randpts)
        im_net = invariant_measure(U_net)
        #L1 = torch.linalg.norm(im_net - im_true, dtype=torch.float32)
        L1 = W2Loss.apply(im_net)
        # L1.backward()
        # optimizer1.step()
        # loss1.append(L1.item())
    # Logging
        with torch.no_grad():
            if i % 500 == 0:
                if L1.item() < 0.05 * initial_L1:
                    break
        L1.backward()
        optimizer1.step()
        loss1.append(L1.item())
    x1 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, 3)
    vals1 = [x1.detach().numpy().flatten()]
    for _ in range(simulation_t-1):
        V_field = Lorenz_vec(x1*M_scale)/M_scale
        V_field[:,invert_variables] = net1(x1)[:,invert_variables]
        x1 = x1+dt* V_field
        vals1.append(x1.detach().numpy().flatten())
    vals1 = np.array(vals1)

    net2 = nn.Sequential(
            nn.Linear(3, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 3))
    optimizer2 = optim.Adam(net2.parameters(),lr = 1e-3)
    net2.train()
    loss2 = []
# Store the first loss
    net2_randpts = randpts.clone()  # safe copy, avoid modifying original tensor
    for _ in range(steps):
        V_field2 = Lorenz_vec(net2_randpts*M_scale)/M_scale
        V_field2[:,invert_variables] = net2(net2_randpts)[:,invert_variables]
        net2_randpts = net2_randpts+dt* V_field2
    initial_L2 = torch.mean((net2_randpts - Trandpts) ** 2)
    for i in range(N_iters):
    # Zero gradients for each optimizer
        optimizer2.zero_grad()
        net2_randpts = randpts
        for _ in range(steps):
            V_field2 = Lorenz_vec(net2_randpts*M_scale)/M_scale
            V_field2[:,invert_variables] = net2(net2_randpts)[:,invert_variables]
            net2_randpts = net2_randpts+dt* V_field2

        L2 = torch.mean((net2_randpts - Trandpts) ** 2)
        # L2.backward()
        # optimizer2.step()
        # loss2.append(L2.item())
        with torch.no_grad():
            if i % 500 == 0:
                if L2.item() < 0.05 * initial_L2:
                    break
        L2.backward()
        optimizer2.step()
        loss2.append(L2.item())
    x2 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, 3)
    vals2 = [x2.detach().numpy().flatten()]
    for _ in range(simulation_t-1):
        V_field2 = Lorenz_vec(x2*M_scale)/M_scale
        V_field2[:,invert_variables] = net2(x2)[:,invert_variables]
        x2 = x2 + V_field2*dt
        vals2.append(x2.detach().numpy().flatten())
    vals2 = np.array(vals2)
    scale = M_scale.detach().cpu().item()
    GT = traj(int(simulation_t), scale * randpts[0].detach().cpu().numpy())
    subvals1 = scale * vals1
    subvals2 = scale * vals2
    a = np.ones(len(GT)) / len(GT)
    b = np.ones(len(subvals1)) / len(subvals1)
    M1 = ot.dist(GT, subvals1, metric="sqeuclidean")
    _, G01 = ot.emd(a, b, M1, log=True)
    cost1W = np.sqrt(G01['cost'])
    M2 = ot.dist(GT, subvals2, metric="sqeuclidean")
    _, G02 = ot.emd(a, b, M2, log=True)
    cost2W = np.sqrt(G02['cost'])
    
    GT_tensor = torch.as_tensor(GT, dtype=randpts.dtype, device=randpts.device)
    GTpts = (GT_tensor / M_scale)[:-steps]
    GTTpts = GT_tensor[steps:]
    with torch.no_grad():
        net1pts, net2pts = GTpts.clone(), GTpts.clone()
        for _ in range(steps):
            Vf1 = Lorenz_vec(net1pts * M_scale) / M_scale
            Vf2 = Lorenz_vec(net2pts * M_scale) / M_scale
            Vf1[:, invert_variables] = net1(net1pts)[:, invert_variables]
            Vf2[:, invert_variables] = net2(net2pts)[:, invert_variables]
            net1pts = net1pts + dt * Vf1
            net2pts = net2pts + dt * Vf2
        cost1L = torch.sqrt(torch.mean((M_scale * net1pts - GTTpts) ** 2))
        cost2L = torch.sqrt(torch.mean((M_scale * net2pts - GTTpts) ** 2))
 
    print("Experiment ", iteration,": W2(ours) ",cost1W, ", W2(pointwise) ",cost2W)
    print("Experiment ", iteration,": L2(ours) ",cost1L.detach().numpy(), ", L2(pointwise) ",cost2L.detach().numpy())
    costs1W.append(cost1W)
    costs2W.append(cost2W)
    costs1L.append(cost1L.detach().numpy())
    costs2L.append(cost2L.detach().numpy())

/tmp/ipykernel_1462219/1883220293.py:43: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x1 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, 3)
/tmp/ipykernel_1462219/1883220293.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x2 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, 3)
/homes/yinonghyn/.local/lib/python3.10/site-packages/ot/lp/__init__.py:388: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  result_code_string = check_result(result_code)


Experiment  0 : W2(ours)  5.756258442671173 , W2(pointwise)  7.1912121655686585
Experiment  0 : L2(ours)  0.12261119 , L2(pointwise)  0.17440501


In [15]:
costs1W,costs2W,costs1L,costs2L

([5.756258442671173,
  5.9296512796467,
  7.344039244074778,
  6.394882229629164,
  5.107195649732987,
  3.8179421416074755,
  4.158050375631227],
 [7.1912121655686585,
  5.253488071882331,
  6.404784989635517,
  7.029492659338601,
  10.932751982869165,
  8.752581681270724,
  5.917222593776135],
 [array(0.12261119, dtype=float32),
  array(0.13526338, dtype=float32),
  array(0.16061644, dtype=float32),
  array(0.13618065, dtype=float32),
  array(0.13259746, dtype=float32),
  array(0.12265339, dtype=float32),
  array(0.11292208, dtype=float32)],
 [array(0.17440501, dtype=float32),
  array(0.21674347, dtype=float32),
  array(0.22910005, dtype=float32),
  array(0.21027324, dtype=float32),
  array(0.22868463, dtype=float32),
  array(0.24868785, dtype=float32),
  array(0.24095918, dtype=float32)])

In [16]:
#W2 error/std (ours)
np.mean(costs1W),np.std(costs1W)

(5.501145623284787, 1.1486636368207246)

In [17]:
##W2 error/std (ptwise)
np.mean(costs2W),np.std(costs2W)

(7.35450487776302, 1.7837676175543378)

In [18]:
#RMSE error/std (ours)
np.mean(costs1L[-5:]),np.std(costs1L[-5:])

(0.132994, 0.016019497)

In [19]:
##RMSE error/std (ptwise)
np.mean(costs2L[-5:]),np.std(costs2L[-5:])

(0.231541, 0.013031894)